# Notebook 2: Linear Regression and Its Types

## Overview
Linear Regression models the relationship between a dependent variable (target) and one or more independent variables (features) using a straight line.

### Types Implemented
| Type | Description |
|---|---|
| **Simple Linear Regression** | One feature → one target |
| **Multiple Linear Regression** | Many features → one target |
| **Polynomial Regression** | Non-linear relationship via polynomial features |
| **Ridge Regression (L2)** | Adds L2 penalty to reduce overfitting |
| **Lasso Regression (L1)** | Adds L1 penalty; performs feature selection |
| **ElasticNet** | Combines L1 + L2 penalties |

## Dataset — House Prices: Advanced Regression Techniques (Kaggle)
**Source:** https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

### How to Download
```bash
# 1. Install the Kaggle API
pip install kaggle

# 2. Place your API token at ~/.kaggle/kaggle.json
#    (Download from https://www.kaggle.com/settings  → API → Create New Token)

# 3. Download & unzip
kaggle competitions download -c house-prices-advanced-regression-techniques -p data/house_prices --unzip
```

The files `train.csv` and `test.csv` will appear in `data/house_prices/`.

In [ ]:
# ─── Install dependencies (uncomment if needed) ───────────────────────────
# !pip install kaggle scikit-learn pandas numpy matplotlib seaborn

# ─── Download dataset from Kaggle ─────────────────────────────────────────
# !kaggle competitions download -c house-prices-advanced-regression-techniques \
#   -p data/house_prices --unzip

## 1. Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
# Path to the downloaded Kaggle dataset
DATA_PATH = 'data/house_prices/train.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
except FileNotFoundError:
    # Fallback: generate a synthetic dataset so the notebook can still run
    print("Kaggle dataset not found – using synthetic data for demonstration.")
    from sklearn.datasets import make_regression
    X_raw, y_raw = make_regression(n_samples=1000, n_features=10, noise=25, random_state=42)
    cols = [f'feature_{i}' for i in range(10)]
    df = pd.DataFrame(X_raw, columns=cols)
    df['SalePrice'] = y_raw + 180_000   # shift to realistic house-price range
    print(f"Synthetic dataset created: {df.shape}")

df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes.value_counts())
print("\nMissing values (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nTarget statistics:")
print(df['SalePrice'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of SalePrice
sns.histplot(df['SalePrice'], kde=True, bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of SalePrice')
axes[0].set_xlabel('SalePrice')

# Log-transformed distribution
sns.histplot(np.log1p(df['SalePrice']), kde=True, bins=40, ax=axes[1], color='salmon')
axes[1].set_title('Distribution of log(SalePrice + 1)')
axes[1].set_xlabel('log(SalePrice + 1)')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Keep only numeric columns for simplicity
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Fill missing numeric values with median
numeric_df.fillna(numeric_df.median(), inplace=True)

# Separate features and target
X = numeric_df.drop(columns=['SalePrice'])
y = np.log1p(numeric_df['SalePrice'])   # log-transform the target (reduces skewness)

print(f"Features shape: {X.shape}")
print(f"Target shape  : {y.shape}")

# Train / validation split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 4. Simple Linear Regression
Uses a **single** feature (GrLivArea – above-ground living area) to predict SalePrice.

In [ ]:
# Select one feature (use first column if 'GrLivArea' not present)
simple_col = 'GrLivArea' if 'GrLivArea' in X.columns else X.columns[0]
print(f"Using feature: {simple_col}")

X_simple = X[[simple_col]]
y_simple = y

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_simple, y_simple, test_size=0.20, random_state=42
)

slr = LinearRegression()
slr.fit(X_tr_s, y_tr_s)
y_pred_s = slr.predict(X_te_s)

print(f"Coefficient : {slr.coef_[0]:.6f}")
print(f"Intercept   : {slr.intercept_:.6f}")
print(f"MSE         : {mean_squared_error(y_te_s, y_pred_s):.6f}")
print(f"R² Score    : {r2_score(y_te_s, y_pred_s):.4f}")

# Plot regression line
plt.figure(figsize=(7, 4))
plt.scatter(X_te_s, y_te_s, alpha=0.4, label='Actual')
sort_idx = X_te_s[simple_col].argsort()
plt.plot(X_te_s.iloc[sort_idx][simple_col], y_pred_s[sort_idx],
         color='red', linewidth=2, label='Regression Line')
plt.xlabel(simple_col)
plt.ylabel('log(SalePrice + 1)')
plt.title('Simple Linear Regression')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Multiple Linear Regression
Uses **all** numeric features.

In [ ]:
mlr = LinearRegression()
mlr.fit(X_train_sc, y_train)
y_pred_m = mlr.predict(X_test_sc)

print("=== Multiple Linear Regression ===")
print(f"MSE  : {mean_squared_error(y_test, y_pred_m):.6f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred_m):.6f}")
print(f"R²   : {r2_score(y_test, y_pred_m):.4f}")

# Residual plot
residuals = y_test - y_pred_m
plt.figure(figsize=(7, 4))
plt.scatter(y_pred_m, residuals, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted log(SalePrice)')
plt.ylabel('Residuals')
plt.title('Multiple Linear Regression – Residual Plot')
plt.tight_layout()
plt.show()

## 6. Polynomial Regression
Models a **non-linear** relationship by adding polynomial feature terms.

In [ ]:
poly_pipeline = Pipeline([
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

# Use a small subset of features to avoid memory issues
top_cols = X.columns[:5].tolist()
poly_pipeline.fit(X_train[top_cols], y_train)
y_pred_poly = poly_pipeline.predict(X_test[top_cols])

print("=== Polynomial Regression (degree=2) ===")
print(f"MSE  : {mean_squared_error(y_test, y_pred_poly):.6f}")
print(f"R²   : {r2_score(y_test, y_pred_poly):.4f}")

## 7. Ridge Regression (L2 Regularisation)
Penalises large coefficients – useful when features are correlated (multicollinearity).

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_sc, y_train)
    y_pred_r = ridge.predict(X_test_sc)
    ridge_results.append({
        'alpha': alpha,
        'MSE': mean_squared_error(y_test, y_pred_r),
        'R2':  r2_score(y_test, y_pred_r)
    })

ridge_df = pd.DataFrame(ridge_results)
print("=== Ridge Regression ===")
print(ridge_df.to_string(index=False))

## 8. Lasso Regression (L1 Regularisation)
Drives some coefficients to **exactly zero**, acting as automatic feature selection.

In [ ]:
lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_sc, y_train)
    y_pred_l = lasso.predict(X_test_sc)
    n_zero = np.sum(lasso.coef_ == 0)
    lasso_results.append({
        'alpha': alpha,
        'MSE': mean_squared_error(y_test, y_pred_l),
        'R2':  r2_score(y_test, y_pred_l),
        'Zeroed features': n_zero
    })

lasso_df = pd.DataFrame(lasso_results)
print("=== Lasso Regression ===")
print(lasso_df.to_string(index=False))

## 9. ElasticNet Regression
Combines both L1 and L2 penalties via the `l1_ratio` parameter.

In [ ]:
en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
en.fit(X_train_sc, y_train)
y_pred_en = en.predict(X_test_sc)

print("=== ElasticNet Regression ===")
print(f"MSE  : {mean_squared_error(y_test, y_pred_en):.6f}")
print(f"R²   : {r2_score(y_test, y_pred_en):.4f}")

## 10. Comparison of All Models

In [ ]:
models = {
    'Multiple Linear Regression': mlr.predict(X_test_sc),
    'Ridge (alpha=1)':            Ridge(alpha=1).fit(X_train_sc, y_train).predict(X_test_sc),
    'Lasso (alpha=0.01)':         Lasso(alpha=0.01, max_iter=5000).fit(X_train_sc, y_train).predict(X_test_sc),
    'ElasticNet (alpha=0.1)':     y_pred_en,
}

comparison = []
for name, preds in models.items():
    comparison.append({
        'Model': name,
        'MSE':   round(mean_squared_error(y_test, preds), 6),
        'MAE':   round(mean_absolute_error(y_test, preds), 6),
        'R²':    round(r2_score(y_test, preds), 4)
    })

print(pd.DataFrame(comparison).to_string(index=False))

## Summary

| Model | Best when… |
|---|---|
| Simple Linear Regression | Clear linear relationship with one predictor |
| Multiple Linear Regression | Many numeric predictors, no strong multicollinearity |
| Polynomial Regression | Non-linear data patterns |
| Ridge (L2) | Multicollinearity; keeps all features |
| Lasso (L1) | Sparse data; many irrelevant features |
| ElasticNet | Grouped correlated features |